# **[ROS2 SLAM과 AI 로봇 네비게이션]** 수업 가이드<br><br>
---
---
<br>

## 0. 비상용

```bash
pkill -9 gz
pkill -9 gz sim
pkill -9 ruby
```

---

## 1. ROS2 Jazzy 설치

```bash
sudo apt install software-properties-common
sudo add-apt-repository universe
sudo apt install ros-dev-tools

sudo apt update
sudo apt upgrade
sudo apt install ros-jazzy-desktop
```

---

## 2. 패키지 설치

```bash
sudo apt update
sudo apt install ros-jazzy-tf2-ros \
                 ros-jazzy-tf-transformations \
                 ros-jazzy-rmw-cyclonedds-cpp \
                 ros-jazzy-turtlebot3-gazebo \
                 ros-jazzy-turtlebot3-description \
                 ros-jazzy-turtlebot3-teleop \
                 ros-jazzy-ros-gz \
                 ros-jazzy-ros-gz-sim \
                 ros-jazzy-ros-gz-bridge \
                 ros-jazzy-rviz2 \
                 ros-jazzy-slam-toolbox \
                 ros-jazzy-nav2-map-server \
                 ros-jazzy-navigation2 ros-jazzy-nav2-bringup
```

---

## 3. ROS2 개발 환경 설정

```bash
echo "source /opt/ros/jazzy/setup.bash" >> ~/.bashrc
source ~/.bashrc
```

---

## 4. ROS2 통신 환경 설정

```bash
echo "export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp" >> ~/.bashrc
echo "export ROS_DOMAIN_ID=0" >> ~/.bashrc
source ~/.bashrc
```

---

## 5. colcon 환경 설정
```bash
mkdir -p ~/colcon_ws/src
cd ~/colcon_ws
colcon build
echo "source ~/colcon_ws/install/setup.bash" >> ~/.bashrc
```

---

## 6. colcon 패키지 생성

**실행 디렉토리:** `~/colcon_ws/src`
```bash
ros2 pkg create pkg_name --build-type ament_python --dependencies rclpy std_msgs geometry_msgs sensor_msgs nav_msgs
```

---

## 7. colcon 패키지 빌드

**실행 디렉토리:** `~/colcon_ws`
```bash
colcon build
```
```bash
colcon build --packages-select pkg_name
```

---

## 8. 빌드 후 Source

**실행 디렉토리:** `~/colcon_ws`
```bash
source install/setup.bash
```

**혹은 아무 디렉토리에서:**
```bash
source ~/colcon/install/setup.bash
```

---

## 9. Demo Launch 파일

```python
from launch import LaunchDescription
from launch.actions import ExecuteProcess, RegisterEventHandler
from launch.event_handlers import OnProcessStart


def generate_launch_description():

    publisher = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name py_name; '
            'exec bash'
        ],
        output='screen'
    )

    processor = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name py_name; '
            'exec bash'
        ],
        output='screen'
    )

    subscriber = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name py_name; '
            'exec bash'
        ],
        output='screen'
    )

    return LaunchDescription([
        publisher,

        RegisterEventHandler(
            OnProcessStart(
                target_action=publisher,
                on_start=[processor]
            )
        ),

        RegisterEventHandler(
            OnProcessStart(
                target_action=processor,
                on_start=[subscriber]
            )
        ),
    ])
```

---

## 10. 커스텀 메세지/서비스

**실행 디렉토리:** `~/colcon_ws/src`
```bash
ros2 pkg create pkg_name --build-type ament_cmake --dependencies rclpy std_msgs geometry_msgs sensor_msgs nav_msgs
```

---

## 11. CMakeLists.txt (전체 복사)
```cmake
cmake_minimum_required(VERSION 3.8)
project(PKG_NAME)

find_package(ament_cmake REQUIRED)
find_package(rosidl_default_generators REQUIRED)

rosidl_generate_interfaces(${PROJECT_NAME}
  "msg/MSG_NAME.msg"
  "srv/SRV_NAME.srv"
)

ament_package()
```

---

## 12. interfaces/package.xml (코드 추가)
```xml
  <build_depend>rosidl_default_generators</build_depend>
  <exec_depend>rosidl_default_runtime</exec_depend>
  <member_of_group>rosidl_interface_packages</member_of_group>
```

---

## 13. my_pkg/package.xml (코드 추가)
```xml
  <depend>interfaces_pkg_name</depend>
```

---

## 14. Turtlebot 환경 변수 설정

```bash
echo "export TURTLEBOT3_MODEL=burger" >> ~/.bashrc
source ~/.bashrc
```

---

## 15. Turtlebot 실행

```bash
ros2 launch turtlebot3_gazebo turtlebot3_dqn_stage1.launch.py
ros2 run turtlebot3_teleop teleop_keyboard
```

---

## 16. 맵핑:

```bash
ros2 launch slam_toolbox online_async_launch.py
```

---

## 17. Turtlebot LiDAR 노이즈 수정

```bash
sudo nano /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf
```

```xml
      <sensor name="hls_lfcd_lds" type="gpu_lidar">
        <always_on>true</always_on>
        <visualize>true</visualize>
        <pose>-0.032 0 0.171 0 0 0</pose>
        <update_rate>5</update_rate>
        <topic>scan</topic>
        <gz_frame_id>base_scan</gz_frame_id>
        <lidar>
          <scan>
            <horizontal>
              <samples>360</samples>
              <resolution>1.000000</resolution>
              <min_angle>0.000000</min_angle>
              <max_angle>6.280000</max_angle>
            </horizontal>
          </scan>
          <range>
            <min>0.120000</min>
            <max>3.5</max>
            <resolution>0.015000</resolution>
          </range>
          <noise>
            <type>gaussian</type>
            <mean>0.0</mean>
            <stddev>0.01</stddev>     <!-- 수정 -->
          </noise>
        </lidar>
      </sensor>
    </link>
```

---

## 18. SLAM Launch 파일 예시:

```python
import os

from launch import LaunchDescription
from launch.actions import ExecuteProcess, IncludeLaunchDescription, SetEnvironmentVariable
from launch.launch_description_sources import PythonLaunchDescriptionSource
from launch_ros.substitutions import FindPackageShare
from launch.substitutions import PathJoinSubstitution


def generate_launch_description():

    # 1. Burger 모델
    set_model = SetEnvironmentVariable(
        name='TURTLEBOT3_MODEL',
        value='burger'
    )

    # 2. Turtlebot3 Gazebo
    gazebo = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            PathJoinSubstitution([
                FindPackageShare('turtlebot3_gazebo'),
                'launch',
                'turtlebot3_world.launch.py'
            ])
        )
    )

    # 3. SLAM Toolbox
    slam_toolbox = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            PathJoinSubstitution([
                FindPackageShare('slam_toolbox'),
                'launch',
                'online_async_launch.py'
            ])
        )
    )

    # 4. RViz2 실행 (config 포함)
    rviz = ExecuteProcess(
        cmd=[
            'rviz2',
            '-d',
            os.path.expanduser('~/.rviz2/RViz_config_파일_이름.rviz')
        ],
        output='screen'
    )

    # 5. 로봇 주행 노드 실행
    corner_escape = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/colcon_ws/install/setup.bash; '
            'ros2 run 패키지_이름 주행_코드_파일_이름; '
            'exec bash'
        ],
        output='screen'
    )

    return LaunchDescription([
        set_model,
        gazebo,
        slam_toolbox,
        rviz,
        corner_escape,
    ])
```

---

## 19. 맵 저장:

```bash
ros2 run nav2_map_server map_saver_cli -f PATH/지도_파일_이름
```

---

## 20. 맵 해상도 변경:

```bash
sudo nano /opt/ros/jazzy/share/slam_toolbox/config/mapper_params_online_async.yaml
```

**resolution 수정:**

```yaml
    debug_logging: false
    throttle_scans: 1
    transform_publish_period: 0.02 #if 0 never publishes odometry
    map_update_interval: 5.0
    resolution: 0.05   #  <----------- 해상도 변경
    restamp_tf: false
    min_laser_range: 0.0 #for rastering images
    max_laser_range: 20.0 #for rastering images
    minimum_time_interval: 0.5
    transform_timeout: 0.2
    tf_buffer_duration: 30.0
    stack_size_to_use: 40000000 #// program needs a larger stack size to serialize large maps
    enable_interactive_mode: true
```

---

## 21. Gazebo 토픽 확인:

```bash
gz topic -l
gz topic -e -t /토픽_이름
```

---

## 22. AMCL 실행:

```bash
ros2 launch nav2_bringup bringup_launch.py map:=PATH/지도_파일_이름.yaml use_sim_time:=true
```

---

## 23. custom_world.worlds 에 추가:

```xml
      <model name="box_obstacle">
        <static>true</static>
        <pose>-2.0 -2.0 0.25 0 0 0</pose>
        <link name="link">
          <collision name="collision">
            <geometry>
              <box><size>0.7 0.7 0.5</size></box>
            </geometry>
          </collision>
          <visual name="visual">
            <geometry>
              <box><size>0.7 0.7 0.5</size></box>
            </geometry>
            <material>
              <ambient>0.53 0.81 1 1</ambient>
              <diffuse>0.53 0.81 1 1</diffuse>
              <specular>0.1 0.1 0.1 1</specular>
            </material>
          </visual>
        </link>
      </model>
      
      <model name="cylinder_obstacle">
        <static>true</static>
        <pose>1.05 -1 0.4 0 0 0</pose>
        <link name="link">
          <collision name="collision">
            <geometry>
              <cylinder>
                <radius>0.15</radius>
                <length>0.8</length>
              </cylinder>
            </geometry>
          </collision>
          <visual name="visual">
            <geometry>
              <cylinder>
                <radius>0.15</radius>
                <length>0.8</length>
              </cylinder>
            </geometry>
            <material>
              <ambient>0.53 1 0.81 1</ambient>
              <diffuse>0.53 1 0.81 1</diffuse>
              <specular>0.1 0.1 0.1 1</specular>
            </material>
          </visual>
        </link>
      </model>
```

---

## A. Git

**git 최초 실행:**
```bash
git config --global user.email "you@example.com"
```
또는
```bash
git config --global user.name "Your Name"
```
<br>

**로그인 정보 저장:**
```bash
git config --global credential.helper store
```

---

## B. Groot2
https://www.behaviortree.dev/groot/
```bash
cd Downloads
chmod +x Groot2-v1.9.0-linux-installer.run
./Groot2-v1.9.0-linux-installer.run
echo "alias groot2='~/Groot2/bin/groot2'" >> ~/.bashrc
```

---